# Dual hose dataset (L, R only) — folder: `dual_hose_tracker`

Uses `hoseL.csv` + `hoseR.csv` only (ignores T/B if present).

**Policy: both or none** — discard frames with only one of L/R.
Null images (neither) are kept (capped by `max_null_ratio`).

Hose-relative: when hose points up, L may sit right of R on screen — not swapped.

For L+R+T+B see `../quad_hose_tracker/`.


In [1]:
import random
import pickle
import sys
from pathlib import Path
import numpy as np

sys.path.append('../../..')
sys.path.append('/home/wanglab/Programs/tracking/DeepLearningUtils')
sys.path.append('/home/wanglab/Programs/tracking/DeepLearningUtils/src')

from hose_data import load_hose_data, audit_hose_label_order


In [2]:
data_folder = '/mnt/c/Users/wanglab/Desktop/pico-hose/'
output_folder = '/mnt/c/Users/wanglab/Desktop/pico-hose/'

target_resolution = (256, 256)
gaussian_sigma = (11, 11)
random_seed_value = 4
train_split = 0.8
max_null_ratio = 1.0

# Separate from quad pickles
train_pkl = 'training_data_hose_lr.pkl'
test_pkl = 'testing_data_hose_lr.pkl'


In [3]:
print('Orientation mix (Lx>Rx often = hose pointing up):')
for session, stats in audit_hose_label_order(data_folder).items():
    n = max(stats['labeled'], 1)
    print(f"  {session}: labeled={stats['labeled']} Lx>Rx={stats['l_x_gt_r']} ({100*stats['l_x_gt_r']/n:.0f}%)")

training_images, training_image_filenames, training_labels = load_hose_data(
    data_folder,
    target_resolution=target_resolution,
    gaussian_sigma=gaussian_sigma,
    include_null_frames=True,
    require_both=True,
    max_null_ratio=max_null_ratio,
    null_seed=random_seed_value,
)

print(f"Images: {training_images.shape}")
print(f"Labels: {training_labels.shape}  # (N,H,W,2) = L,R")


Orientation mix (Lx>Rx often = hose pointing up):
  102225_2: labeled=234 Lx>Rx=0 (0%)
  102525_1: labeled=176 Lx>Rx=80 (45%)


Loading sessions:   0%|          | 0/2 [00:00<?, ?it/s]

102225_2: 381 images, L=234, R=234, resolution=(480, 640)


Loading sessions:  50%|#####     | 1/2 [00:08<00:08,  8.76s/it]

102525_1: 215 images, L=176, R=176, resolution=(480, 640)


Loading sessions: 100%|##########| 2/2 [00:13<00:00,  6.61s/it]

Loaded 596 frames (both=410, null=186, discarded_partial=0, unreadable=0)
Keypoint channels: ['hoseL', 'hoseR']
Images: (596, 256, 256, 3)
Labels: (596, 256, 256, 2)  # (N,H,W,2) = L,R


In [4]:
has_any = np.any(training_labels > 0, axis=(1, 2, 3))
print(f"Total: {len(training_image_filenames)}")
print(f"Both L+R: {int(has_any.sum())}")
print(f"Null: {int((~has_any).sum())}")


Total: 596
Both L+R: 410
Null: 186


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, interact

def show_example(idx=0):
    img = training_images[idx]
    labels = training_labels[idx]
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    axes[0].imshow(img[..., ::-1])
    axes[0].set_title(Path(training_image_filenames[idx]).name)
    axes[0].axis('off')
    for c, (name, color) in enumerate([('hoseL', 'yellow'), ('hoseR', 'cyan')]):
        ax = axes[c + 1]
        ax.imshow(img[..., ::-1])
        heat = labels[..., c]
        if np.any(heat > 0):
            cy, cx = np.unravel_index(int(np.argmax(heat)), heat.shape)
            ax.scatter(cx, cy, c=color, s=80, marker='x')
            ax.set_title(f'{name} ({cx},{cy})')
        else:
            ax.set_title(f'{name}: none')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

interact(show_example, idx=IntSlider(min=0, max=max(0, len(training_images)-1), value=0))


interactive(children=(IntSlider(value=0, description='idx', max=595), Output()), _dom_classes=('widget-interac…

<function __main__.show_example(idx=0)>

In [6]:
shuffled = list(range(training_images.shape[0]))
random.Random(random_seed_value).shuffle(shuffled)
n_train = int(len(shuffled) * train_split)
train_idx, test_idx = shuffled[:n_train], shuffled[n_train:]

out = Path(output_folder)
out.mkdir(parents=True, exist_ok=True)
labels = np.asarray(training_labels)
assert labels.shape[-1] == 2

with open(out / train_pkl, 'wb') as f:
    pickle.dump((training_images[train_idx], labels[train_idx]), f)
with open(out / test_pkl, 'wb') as f:
    pickle.dump((training_images[test_idx], labels[test_idx]), f)

print(f"Train {len(train_idx)}, Test {len(test_idx)}")
print(f"Wrote {out / train_pkl}")
print(f"Wrote {out / test_pkl}")


Train 476, Test 120
Wrote /mnt/c/Users/wanglab/Desktop/pico-hose/training_data_hose_lr.pkl
Wrote /mnt/c/Users/wanglab/Desktop/pico-hose/testing_data_hose_lr.pkl
